# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tal3at-M/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/Tal3at-M/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 1. Feature Engineering
df["ctr_calc"] = df["clicks_90d"] / (df["impressions_90d"] + 1e-5)
df["log_impressions"] = np.log1p(df["impressions_90d"])
df["log_clicks"] = np.log1p(df["clicks_90d"])

# 2. Selected Feature Set (strictly before-the-fact features)
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "content_age_days",
    "log_impressions",
    "log_clicks",
    "ctr_calc",
]

X = df[feature_cols].copy()
y = (df["trend_direction"].str.lower() == "down").astype(int)

# 3. Handle missing values
X = X.fillna(X.median())

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"Target distribution (1=down, 0=other): {np.bincount(y)}")
X.head(3)

Feature matrix shape: (30000, 7)
Features: ['impressions_90d', 'clicks_90d', 'avg_position', 'content_age_days', 'log_impressions', 'log_clicks', 'ctr_calc']
Target distribution (1=down, 0=other): [13738 16262]


,impressions_90d,clicks_90d,avg_position,content_age_days,log_impressions,log_clicks,ctr_calc
0,3803,29,10.6,187,8.243808,3.401197,0.007626
1,15320,7,20.3,445,9.636980,2.079442,0.000457
2,12581,11,36.5,141,9.440023,2.484907,0.000874


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
#- `impressions_90d`: Total search impressions in the baseline 90-day window. Available before prediction. No missing values.
#- `clicks_90d`: Total organic search clicks in the baseline window. Available before prediction. No missing values.
#- `avg_position`: Mean organic search rank across queries. Available before prediction. No missing values.
#- `content_age_days`: Time elapsed since page publication. Available at prediction time. No missing values.
#- `log_impressions` & `log_clicks`: Log-transformed engagement metrics to normalize power-law distribution. Available before prediction.
#- `ctr_calc`: Calculated click-through rate ratio. Missing/inf guarded by a small epsilon constant.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# Leakage attack test: Checking correlation with the target
correlations = X.apply(lambda col: col.corr(y)).sort_values(ascending=False)
print("--- Correlation with Target (Decay Flag) ---")
print(correlations.round(4))

# Assert no feature perfectly predicts the target (correlation < 0.90)
assert all(
    correlations.abs() < 0.90
), "Potential target leakage detected! A feature is too predictive."
print("\nPass: No direct target leakage detected.")

--- Correlation with Target (Decay Flag) ---
log_impressions     0.1775
log_clicks          0.0035
impressions_90d    -0.0182
avg_position       -0.0290
clicks_90d         -0.0397
ctr_calc           -0.0619
content_age_days   -0.1639
dtype: float64

Pass: No direct target leakage detected.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
#- `trend_direction` / `trend_pct`: Excluded because they directly define or derive the target label (Direct Target Leakage).
#- `impressions_last_30d` / `clicks_last_30d` / `sessions_last_30d`: Excluded because they belong to the post-baseline evaluation period (Temporal Lookahead Leakage).
#- `client_id` / `content_id`: Excluded to prevent client-specific memorization and ensure model generalization across sites.